# Section 1: Introduction

This notebook addresses **R004** by building a physical grey-box thermal model for the HeatSync heating window. Instead of learning temperature dynamics purely from historical correlations as in the baseline and LSTM notebooks, a grey-box model encodes a small thermal circuit whose parameters can be interpreted physically.

The 2R2C analogy represents the space as two thermal states: indoor air temperature and a slower wall/building-mass temperature. The two resistances model heat-transfer bottlenecks between indoor air, wall mass, and outdoor ambient air; the two capacitances model how much thermal energy is required to change indoor-air and wall temperatures. Calibrating these values with positivity bounds keeps the fitted model physically plausible while still using measured data to identify parameters.

The fitted model below is intentionally compact: it is a transparent physical proof point for R004, not a replacement for richer data-driven approaches when occupancy, controls, or unmeasured disturbances dominate.

# Section 2: Data Loading & Physical Input Selection

We load the processed heating dataset and select one device to keep the calibration problem well-defined. The physical inputs are:

- `T_in`: `status_temperature_in_celsius`, the observed indoor temperature.
- `T_out`: `outside_temperature`, the ambient/outdoor temperature engineered by preprocessing.
- `Q`: `power_draw_kw`, used as a heat-input proxy.

`power_draw_kw` is only a proxy for delivered heat: it does not include coefficient-of-performance changes, fan operation, compressor cycling details, solar gains, occupancy gains, or ventilation/infiltration effects. The fitted parameters should therefore be interpreted as effective parameters for this dataset window, not audited equipment specifications.

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# Required by the task plan; useful when launched from backend/notebooks.
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Robust path support for programmatic execution from the repository root.
backend_path = os.path.abspath('backend')
if os.path.isdir(backend_path) and backend_path not in sys.path:
    sys.path.append(backend_path)

from notebooks.utils import load_processed  # noqa: E402

heating_df = load_processed('heating')
first_device = heating_df['device_id'].dropna().iloc[0]
device_df = heating_df.loc[heating_df['device_id'] == first_device].copy()

physical_cols = [
    'status_temperature_in_celsius',
    'outside_temperature',
    'power_draw_kw',
]
model_df = (
    device_df[physical_cols]
    .rename(
        columns={
            'status_temperature_in_celsius': 'T_in',
            'outside_temperature': 'T_out',
            'power_draw_kw': 'Q',
        }
    )
    .dropna()
)

# Bound the fitting window to keep repeated notebook execution fast and deterministic.
model_df = model_df.iloc[:300].copy()
dt = 900.0  # 15-minute sampling interval in seconds
MIN_CALIBRATION_ROWS = 10

if len(model_df) < MIN_CALIBRATION_ROWS:
    raise ValueError(
        f'Need at least {MIN_CALIBRATION_ROWS} complete physical-input rows '
        'for 2R2C calibration.'
    )

T_in = model_df['T_in'].to_numpy(dtype=float)
T_out = model_df['T_out'].to_numpy(dtype=float)
Q = model_df['Q'].to_numpy(dtype=float)

print(f'Device selected: {first_device}')
print(f'Calibration rows: {len(model_df)}')
print(model_df.head())

# Section 3: 2R2C Model Equations

The discrete-time 2R2C model uses state vector `[T_indoor, T_wall]` and positive parameters `R_in`, `C_in`, `R_out`, and `C_wall`:

```text
T_indoor[k+1] = T_indoor[k] + dt/C_in * ((T_wall[k] - T_indoor[k])/R_in + Q[k])
T_wall[k+1]   = T_wall[k]   + dt/C_wall * ((T_out[k] - T_wall[k])/R_out - (T_wall[k] - T_indoor[k])/R_in)
```

`R_in` and `R_out` control resistance to heat flow, while `C_in` and `C_wall` control thermal inertia. All four parameters are constrained to positive values during calibration. Because the dataset records `Q` in kW and `dt` is stored as seconds, the implementation converts `dt` to hours inside the simulator for unit consistency while preserving the task-required `dt = 900.0` seconds input.

In [ ]:
SECONDS_PER_HOUR = 3_600.0
MAX_ABSOLUTE_TEMPERATURE = 1_000


def simulate_2r2c(params, T_in_init, T_wall_init, T_out, Q, dt):
    """Simulate indoor temperature for a discrete 2R2C model."""
    R_in, C_in, R_out, C_wall = params
    if np.any(np.asarray(params, dtype=float) <= 0):
        raise ValueError('2R2C parameters must all be positive.')

    dt_hours = dt / SECONDS_PER_HOUR
    n_steps = len(T_out)
    T_pred = np.empty(n_steps, dtype=float)
    T_indoor = float(T_in_init)
    T_wall = float(T_wall_init)
    T_pred[0] = T_indoor

    for k in range(n_steps - 1):
        indoor_delta = dt_hours / C_in * (((T_wall - T_indoor) / R_in) + Q[k])
        wall_delta = dt_hours / C_wall * (
            ((T_out[k] - T_wall) / R_out) - ((T_wall - T_indoor) / R_in)
        )
        T_indoor = T_indoor + indoor_delta
        T_wall = T_wall + wall_delta

        if not np.isfinite(T_indoor) or not np.isfinite(T_wall):
            T_pred[k + 1 :] = np.nan
            break

        # Guard against unstable candidate parameters dominating optimizer time.
        if (
            abs(T_indoor) > MAX_ABSOLUTE_TEMPERATURE
            or abs(T_wall) > MAX_ABSOLUTE_TEMPERATURE
        ):
            T_pred[k + 1 :] = np.nan
            break

        T_pred[k + 1] = T_indoor

    return T_pred


# Section 4: Bounded L-BFGS-B Calibration

The objective minimizes mean squared error between the simulated indoor temperature and observed `T_in`. L-BFGS-B supports simple bounds, so each parameter is constrained to `(0.001, 50.0)` to enforce positivity. Multiple deterministic random restarts reduce sensitivity to initialization.

In [ ]:
T_wall_init = (T_in[0] + T_out[0]) / 2
bounds = [(0.001, 50.0)] * 4
param_names = ['R_in', 'C_in', 'R_out', 'C_wall']
UNSTABLE_OBJECTIVE_PENALTY = 1e12


def objective(params):
    T_pred = simulate_2r2c(params, T_in[0], T_wall_init, T_out, Q, dt)
    if not np.all(np.isfinite(T_pred)):
        return UNSTABLE_OBJECTIVE_PENALTY
    return float(np.mean((T_pred - T_in) ** 2))


np.random.seed(42)
restart_results = []
for restart_id in range(8):
    x0 = np.random.uniform(0.1, 25.0, size=4)
    result = minimize(
        objective,
        x0=x0,
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 500, 'ftol': 1e-10},
    )
    restart_results.append(result)
    print(
        f'Restart {restart_id + 1}: success={result.success} '
        f'status={result.status} fun={result.fun:.6f}'
    )

best_result = min(restart_results, key=lambda res: res.fun)
best_params = best_result.x
converged_restarts = [idx + 1 for idx, res in enumerate(restart_results) if res.success]

print('\nBest optimizer status:', best_result.status)
print('Best optimizer message:', best_result.message)
print(f'Best objective MSE: {best_result.fun:.6f}')
print('Converged restarts:', converged_restarts)

# Section 5: 2R2C Results

The table and plot below summarize the calibrated positive physical parameters and the fitted indoor-temperature trajectory. The resulting parameters are effective thermal-circuit values for this measured window and proxy heat input.

In [ ]:
param_table = pd.DataFrame(
    {'parameter': param_names, 'value': best_params}
).set_index('parameter')

assert (param_table['value'] > 0).all(), 'All calibrated 2R2C parameters must be positive.'
param_table

In [ ]:
T_fit = simulate_2r2c(best_params, T_in[0], T_wall_init, T_out, Q, dt)
assert np.all(np.isfinite(T_fit)), 'Calibrated 2R2C fit must be finite.'
rmse = float(np.sqrt(np.mean((T_fit - T_in) ** 2)))
assert np.isfinite(rmse), 'Calibrated 2R2C RMSE must be finite.'

print(f'2R2C calibrated RMSE: {rmse:.4f} °C')

plt.figure(figsize=(12, 5))
plt.plot(model_df.index, T_in, label='Observed indoor temperature', linewidth=2)
plt.plot(model_df.index, T_fit, label='2R2C fitted temperature', linewidth=2)
plt.xlabel('Timestamp')
plt.ylabel('Indoor temperature (°C)')
plt.title('2R2C Fitted vs Observed Indoor Temperature')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Interpretation

Positive `R_in`, `C_in`, `R_out`, and `C_wall` values confirm that the optimizer found a physically admissible 2R2C parameter set. Smaller resistance values imply easier heat transfer across the corresponding boundary; larger capacitance values imply slower temperature movement for the same heat flow.

The model is deliberately simple. It treats electrical power draw as a delivered-heat proxy, assumes a fixed 15-minute step, and omits occupancy gains, solar gains, ventilation, defrost cycles, compressor efficiency changes, and controller state. Those omissions can bias effective parameters, but the bounded calibration still provides an interpretable physical baseline for R004.